In [2]:
!pip -q install tqdm scikit-learn

In [ ]:


import os, re, json, pickle, math
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizerFast, BertModel, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score

# --------------------------
# Device
# --------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


BASE = "/kaggle/input/piash-dpfn-mabsa-data"

TRAIN_TSV = f"{BASE}/train.tsv"
DEV_TSV   = f"{BASE}/dev.tsv"
TEST_TSV  = f"{BASE}/test.tsv"

CAPTION_JSON = f"{BASE}/twitter2015_images.json"
VIT_PKL      = f"{BASE}/twitter2015_images_clip_vitb16.pkl"

# Original syntax adjacency (word-word)
TRAIN_ORI = f"{BASE}/train_ori_adj.pkl"
DEV_ORI   = f"{BASE}/dev_ori_adj.pkl"
TEST_ORI  = f"{BASE}/test_ori_adj.pkl"

# Updated affective adjacency (word-word)
TRAIN_AFF = f"{BASE}/train_aff_adj_updated.pkl"
DEV_AFF   = f"{BASE}/dev_aff_adj_updated.pkl"
TEST_AFF  = f"{BASE}/test_aff_adj_updated.pkl"

# Updated word+concept adjacency (NO sentence node)
TRAIN_WC_ADJ = f"{BASE}/train_word_concept_adj_k5_updated.pkl"
DEV_WC_ADJ   = f"{BASE}/dev_word_concept_adj_k5_updated.pkl"
TEST_WC_ADJ  = f"{BASE}/test_word_concept_adj_k5_updated.pkl"

# Updated word+concept node features (BERT) aligned to that adjacency
TRAIN_WC_NF = f"{BASE}/train_word_concept_nodefeats_k5_updated.pkl"
DEV_WC_NF   = f"{BASE}/dev_word_concept_nodefeats_k5_updated.pkl"
TEST_WC_NF  = f"{BASE}/test_word_concept_nodefeats_k5_updated.pkl"

# Optional (not needed to train, but you can load if you want)
TRAIN_WC_NAMES = f"{BASE}/train_word_concept_nodenames_k5_updated.pkl"
DEV_WC_NAMES   = f"{BASE}/dev_word_concept_nodenames_k5_updated.pkl"
TEST_WC_NAMES  = f"{BASE}/test_word_concept_nodenames_k5_updated.pkl"

TRAIN_TOPK = f"{BASE}/train_topk_concepts_k5_updated.pkl"
DEV_TOPK   = f"{BASE}/dev_topk_concepts_k5_updated.pkl"
TEST_TOPK  = f"{BASE}/test_topk_concepts_k5_updated.pkl"



OUT_DIR = "/kaggle/working/ke_gcn_runs"
os.makedirs(OUT_DIR, exist_ok=True)
print("OUT_DIR:", OUT_DIR)

# --------------------------
# Options
# --------------------------
class Opt:
    MAX_LEN = 50
    BATCH_SIZE = 32
    NUM_CLASSES = 3
    EPOCHS = 30
    LEARNING_RATE = 5e-5
    NUM_WARMUP_STEPS = 0
    NUM_RUNS = 1
    DEVICE = device
    K_CONCEPT = 5
opt = Opt()

# --------------------------
# Helpers
# --------------------------
def read_tsv(path, split):
    df = pd.read_csv(path, sep="\t")
    if split in ("train","dev"):
        df = df.rename({
            "#1 Label": "sentiment",
            "#2 ImageID": "image_id",
            "#3 String": "tweet_content",
            "#3 String.1": "target",
        }, axis=1).drop(["index"], axis=1)
    else:
        df = df.rename({
            "index": "sentiment",
            "#1 ImageID": "image_id",
            "#2 String": "tweet_content",
            "#2 String.1": "target",
        }, axis=1)
    return df

def load_pkl(p):
    with open(p, "rb") as f:
        return pickle.load(f)

# --------------------------
# Load TSVs / Caption / ViT
# --------------------------
train_df = read_tsv(TRAIN_TSV, "train")
dev_df   = read_tsv(DEV_TSV,   "dev")
test_df  = read_tsv(TEST_TSV,  "test")
print("rows:", len(train_df), len(dev_df), len(test_df))

with open(CAPTION_JSON, "r") as f:
    captions = json.load(f)
print("captions:", len(captions))

with open(VIT_PKL, "rb") as f:
    vit_feats = pickle.load(f)
print("vit feats:", len(vit_feats))

# --------------------------
# Load graphs/features PKLs
# --------------------------
train_ori = load_pkl(TRAIN_ORI)
dev_ori   = load_pkl(DEV_ORI)
test_ori  = load_pkl(TEST_ORI)

train_aff = load_pkl(TRAIN_AFF)
dev_aff   = load_pkl(DEV_AFF)
test_aff  = load_pkl(TEST_AFF)

train_wc_adj = load_pkl(TRAIN_WC_ADJ)
dev_wc_adj   = load_pkl(DEV_WC_ADJ)
test_wc_adj  = load_pkl(TEST_WC_ADJ)

train_wc_nf = load_pkl(TRAIN_WC_NF)
dev_wc_nf   = load_pkl(DEV_WC_NF)
test_wc_nf  = load_pkl(TEST_WC_NF)

print("train sizes ori/aff/wc_adj/wc_nf:",
      len(train_ori), len(train_aff), len(train_wc_adj), len(train_wc_nf))


class MyTokenizer:
    def __init__(self, max_len):
        self.tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")
        self.max_len = max_len

    def text_to_sequence(self, text, aspect, tran=False):
        left, right = [s.strip() for s in text.split("$T$")]

        # target mask at word level (whitespace words)
        target_mask = len(left.split()) * [0] + len(aspect.split()) * [1] + len(right.split()) * [0]
        target_mask = torch.tensor([0] + target_mask + (self.max_len - 1 - len(target_mask)) * [0])

        words = (left + " " + aspect + " " + right).split()
        word_len = len(words)

        trans = []
        realwords = []
        for w in words:
            pieces = self.tokenizer.tokenize(w)
            start = len(realwords)
            realwords.extend(pieces)
            trans.append([start, len(realwords)])

        # pad trans to MAX_LEN
        start_num = len(realwords)
        for _ in range(len(trans), self.max_len):
            trans.append([start_num, start_num+1])
            start_num += 1

        ids = [self.tokenizer.cls_token_id] + [self.tokenizer.convert_tokens_to_ids(w) for w in realwords] + [self.tokenizer.sep_token_id]

        if len(ids) <= self.max_len:
            input_ids = torch.tensor(ids + (self.max_len - len(ids)) * [0])
            attn = torch.tensor(len(ids) * [1] + (self.max_len - len(ids)) * [0])
        else:
            input_ids = torch.tensor(ids[:self.max_len])
            attn = torch.tensor(self.max_len * [1])

        text_length = len(ids)
        if tran:
            return input_ids, attn, text_length, target_mask, word_len, trans
        return input_ids, attn, text_length, target_mask, word_len

# --------------------------

2026-01-29 14:42:41.481463: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769697761.678660      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769697761.736970      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769697762.231291      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769697762.231329      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769697762.231332      24 computation_placer.cc:177] computation placer alr

device: cuda
OUT_DIR: /kaggle/working/dpfn_runs_updated
rows: 3179 1122 1037
captions: 8288
vit feats: 8288
train sizes ori/aff/wc_adj/wc_nf: 3179 3179 3179 3179


In [ ]:
# --------------------------
# Dataset 
# --------------------------
class Twitter2015KEGCN(Dataset):
    """
    Returns:
      - ori_adj:  word-word  [N,N] padded into [MAX_LEN,MAX_LEN]
      - aff_adj:  word-word  [N,N] padded into [MAX_LEN,MAX_LEN]
      - wc_adj:   word+concept [(N+K),(N+K)] padded into [MAX_LEN+K, MAX_LEN+K]
      - wc_node_feats: precomputed [MAX_LEN+K,768] from PKL (already padded/aligned)
    """
    def __init__(self, df, split_name, max_len, captions, vit_feats,
                 ori_adj, aff_adj, wc_adj, wc_node_feats, k_con=5):
        self.df = df.reset_index(drop=True)
        self.max_len = max_len
        self.captions = captions
        self.vit_feats = vit_feats

        self.ori_adj = ori_adj
        self.aff_adj = aff_adj

        self.wc_adj = wc_adj
        self.wc_node_feats = wc_node_feats

        self.k_con = k_con

        self.tokenizer_self = MyTokenizer(max_len)
        self.pair_tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

        print(f"{split_name} size:", len(self.df))

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        tweet = row["tweet_content"]
        aspect = row["target"]
        label = int(row["sentiment"])
        image_id = row["image_id"]

        cap = self.captions.get(image_id, "")
        vit = self.vit_feats[image_id]
        if isinstance(vit, np.ndarray):
            vit = torch.tensor(vit, dtype=torch.float32)
        else:
            vit = vit.float()

        # global branch inputs (tweet-with-aspect paired with caption)
        enc = self.pair_tokenizer.encode_plus(
            tweet.replace("$T$", aspect),
            text_pair=cap,
            add_special_tokens=True,
            max_length=80,
            return_attention_mask=True,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        globel_input_id = enc["input_ids"].flatten()
        globel_mask     = enc["attention_mask"].flatten()

        # local branch inputs (repo tokenizer)
        input_ids, attn, text_length, target_mask, word_len, trans = self.tokenizer_self.text_to_sequence(tweet, aspect, tran=True)

        # ===== pad ori + aff into [MAX_LEN,MAX_LEN] (word-level) =====
        A_ori = np.array(self.ori_adj[i], dtype=np.float32)
        A_aff = np.array(self.aff_adj[i], dtype=np.float32)

        ctx_ori = np.zeros((self.max_len, self.max_len), dtype=np.float32)
        ctx_aff = np.zeros((self.max_len, self.max_len), dtype=np.float32)

        ctx_ori[1:word_len+1, 1:word_len+1] = A_ori
        ctx_aff[1:word_len+1, 1:word_len+1] = A_aff

        # ===== word+concept adjacency padded into [MAX_LEN+K, MAX_LEN+K] =====
        # Your saved wc_adj is in raw size (N+K)x(N+K).
        A_wc_raw = np.array(self.wc_adj[i], dtype=np.float32)
        L = self.max_len + self.k_con
        ctx_wc = np.zeros((L, L), dtype=np.float32)

        # word-word block (put into [1..word_len] like before)
        ctx_wc[1:word_len+1, 1:word_len+1] = A_wc_raw[:word_len, :word_len]

        # concept nodes are stored at indices [word_len .. word_len+K-1] in raw
        # we place them at [MAX_LEN .. MAX_LEN+K-1] in padded
        for c in range(self.k_con):
            src_c = word_len + c
            dst_c = self.max_len + c
            if src_c < A_wc_raw.shape[0]:
                ctx_wc[1:word_len+1, dst_c] = A_wc_raw[:word_len, src_c]
                ctx_wc[dst_c, 1:word_len+1] = A_wc_raw[src_c, :word_len]
                ctx_wc[dst_c, dst_c] = 1.0

         # ===== node features from PKL =====
        # raw: (word_len+K, 768) aligned to raw indexing:
        #   words:   [0..word_len-1]
        #   concepts:[word_len..word_len+K-1]
        X_raw = self.wc_node_feats[i]
        if isinstance(X_raw, torch.Tensor):
            X_raw = X_raw.float()
        else:
            X_raw = torch.tensor(X_raw, dtype=torch.float32)
        
        # build padded node feats: (MAX_LEN+K, 768)
        L = self.max_len + self.k_con
        X_pad = torch.zeros((L, 768), dtype=torch.float32)
        
        # (1) words -> positions [1..word_len]
        X_pad[1:word_len+1, :] = X_raw[:word_len, :]
        
        # (2) concepts -> positions [MAX_LEN..MAX_LEN+K-1]
        for c in range(self.k_con):
            src_c = word_len + c
            dst_c = self.max_len + c
            if src_c < X_raw.size(0):
                X_pad[dst_c, :] = X_raw[src_c, :]
        
        wc_nf = X_pad


        # transformer_mask kept for compatibility (repo style)
        transformer_mask = (attn == 0).numpy().tolist() + (attn == 0).numpy().tolist()
        transformer_mask = torch.tensor(transformer_mask)

        return {
            "input_ids": input_ids,
            "attention_mask": attn,
            "targets": torch.tensor(label, dtype=torch.long),
            "vit_feature": vit,
            "transformer_mask": transformer_mask,

            "target_mask": target_mask,
            "text_length": torch.tensor(text_length),
            "word_length": torch.tensor(word_len),
            "tran_indices": torch.tensor(trans),

            "context_asp_adj_matrix": torch.tensor(ctx_ori),
            "context_aff_adj_matrix": torch.tensor(ctx_aff),

            # UPDATED concept graph
            "context_wc_adj_matrix": torch.tensor(ctx_wc),
            "wc_node_feats": wc_nf,

            "globel_input_id": globel_input_id,
            "globel_mask": globel_mask,
        }

def make_loader(df, split, batch_size, shuffle, ori, aff, wc_adj, wc_nf):
    ds = Twitter2015KEGCN(
        df, split, opt.MAX_LEN, captions, vit_feats,
        ori_adj=ori, aff_adj=aff, wc_adj=wc_adj, wc_node_feats=wc_nf,
        k_con=opt.K_CONCEPT
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0)

train_loader = make_loader(train_df, "train", opt.BATCH_SIZE, True,  train_ori, train_aff, train_wc_adj, train_wc_nf)
dev_loader   = make_loader(dev_df,   "dev",   opt.BATCH_SIZE, False, dev_ori,   dev_aff,   dev_wc_adj,   dev_wc_nf)
test_loader  = make_loader(test_df,  "test",  opt.BATCH_SIZE, False, test_ori,  test_aff,  test_wc_adj,  test_wc_nf)

# --------------------------

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

train size: 3179
dev size: 1122
test size: 1037


In [ ]:
# --------------------------
# Model (MHSA-based similarity using existing MultiHeadAttention)

# --------------------------
class MultiHeadAttention(nn.Module):
    def __init__(self, n_head, d_model):
        super().__init__()
        self.n_head = n_head
        self.dk = d_model
        self.wq = nn.Linear(d_model, n_head * d_model, bias=False)
        self.wk = nn.Linear(d_model, n_head * d_model, bias=False)
        self.wv = nn.Linear(d_model, n_head * d_model, bias=False)
        self.fc = nn.Linear(n_head * d_model, d_model, bias=False)
        self.ln = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(0.1)

    def forward(self, q, k, v, attn_mask=None):
        B, Lq, D = q.size()
        B, Lk, D = k.size()

        qh = self.wq(q).view(B, Lq, self.n_head, D).permute(2,0,1,3).contiguous().view(-1, Lq, D)
        kh = self.wk(k).view(B, Lk, self.n_head, D).permute(2,0,1,3).contiguous().view(-1, Lk, D)
        vh = self.wv(v).view(B, Lk, self.n_head, D).permute(2,0,1,3).contiguous().view(-1, Lk, D)

        attn = torch.bmm(qh, kh.transpose(1,2)) / math.sqrt(D)
        if attn_mask is not None:
            attn = attn.masked_fill(attn_mask, -1e6)
        attn = torch.softmax(attn, dim=-1)
        out = torch.bmm(attn, vh)

        out = out.view(self.n_head, B, Lq, D).permute(1,2,0,3).contiguous().view(B, Lq, self.n_head*D)
        out = self.fc(out)
        out = self.drop(out)
        out = self.ln(out + q)
        return out

class KEGCN(nn.Module):
    def __init__(self, opt):
        super().__init__()
        self.opt = opt
        self.max_len = opt.MAX_LEN
        self.k_con = opt.K_CONCEPT
        self.layers = 2

        self.bert = BertModel.from_pretrained("bert-base-uncased")
        self.caption_text_bert = BertModel.from_pretrained("bert-base-uncased")

        self.layernorm = nn.LayerNorm(768)
        self.drop = nn.Dropout(0.1)

        self.text_img_attn = MultiHeadAttention(n_head=8, d_model=768)

        # NEW: reuse same class for self-attn similarity projection
        self.self_attn_sim = MultiHeadAttention(n_head=8, d_model=768)

        self.W_aff = nn.ModuleList([nn.Linear(768,768) for _ in range(self.layers)])
        self.W_con = nn.ModuleList([nn.Linear(768,768) for _ in range(self.layers)])

        self.W_faff = nn.ModuleList([nn.Linear(768,768) for _ in range(self.layers)])
        self.W_fcon = nn.ModuleList([nn.Linear(768,768) for _ in range(self.layers)])

        self.out = nn.Linear(5*768, opt.NUM_CLASSES)

    def gcn_dense(self, X, A, Ws):
        #denom = A.sum(dim=2, keepdim=True) + 1.0
        denom = A.abs().sum(dim=2, keepdim=True) + 1.0 #changed due to value getting lardge after multiplying sim

        H = X
        for l in range(self.layers):
            Ax = torch.bmm(A, H)
            AxW = Ws[l](Ax) / denom
            H = F.relu(AxW)
        return H

    def forward(self, batch):
        input_ids = batch["input_ids"]
        attn_mask = batch["attention_mask"]
        vit = batch["vit_feature"]
        target_mask = batch["target_mask"]
        word_len = batch["word_length"]
        tran_indices = batch["tran_indices"]

        A_aff = batch["context_aff_adj_matrix"].float()
        A_con_full = batch["context_wc_adj_matrix"].float()
        wc_node_feats = batch["wc_node_feats"].float()

        globel_input_id = batch["globel_input_id"]
        globel_mask = batch["globel_mask"]

        out = self.bert(input_ids=input_ids, attention_mask=attn_mask).last_hidden_state
        out = self.drop(self.layernorm(out))

        txt_img = self.text_img_attn(out, vit, vit)   # [B,MAX_LEN,768]

        # -------- REPLACE COSINE SIM WITH MHSA-BASED SIM --------
        # self-attend on txt_img (q=k=v=txt_img), then similarity = dot product of attended tokens
        # txt_self = self.self_attn_sim(txt_img, txt_img, txt_img)  # [B,MAX_LEN,768]
        # sim = torch.bmm(txt_self, txt_self.transpose(1, 2)) / math.sqrt(txt_self.size(-1))  # [B,MAX_LEN,MAX_LEN]

           
        txt_self = self.self_attn_sim(txt_img, txt_img, txt_img)
        txt_self = F.normalize(txt_self, dim=-1)
        sim = torch.bmm(txt_self, txt_self.transpose(1,2))   # [-1,1]

        
        # --------------------------------------------------------

        global_bert = self.caption_text_bert(input_ids=globel_input_id, attention_mask=globel_mask).pooler_output
        global_bert = self.drop(self.layernorm(global_bert))

        gcn_text_feat = out[:, 1:, :]
        gcn_text_img_feat = txt_img[:, 1:, :]

        B = input_ids.size(0)
        tmps = torch.zeros((B, self.max_len, 768), device=input_ids.device)
        tmps_img = torch.zeros((B, self.max_len, 768), device=input_ids.device)

        for i, spans in enumerate(tran_indices):
            true_len = int(word_len[i].item())
            nums = 0
            for j, span in enumerate(spans):
                if nums == true_len:
                    break
                nums += 1
                a, b = int(span[0].item()), int(span[1].item())
                tmps[i, j+1] = torch.sum(gcn_text_feat[i, a:b], dim=0)
                tmps_img[i, j+1] = torch.sum(gcn_text_img_feat[i, a:b], dim=0)

        H_aff = self.gcn_dense(tmps, A_aff, self.W_aff)

        asp_wn = target_mask.sum(dim=1, keepdim=True) + 1e-6
        asp_mask = target_mask.unsqueeze(-1).repeat(1, 1, 768)

        con_feats = wc_node_feats[:, self.max_len:self.max_len+self.k_con, :]
        X_con = torch.cat([tmps, con_feats], dim=1)
        H_con = self.gcn_dense(X_con, A_con_full, self.W_con)

        A_faff = A_aff * sim
        H_faff = self.gcn_dense(tmps_img, A_faff, self.W_faff)

        A_fcon_full = A_con_full.clone()
        A_con_words = A_con_full[:, :self.max_len, :self.max_len]
        A_fcon_full[:, :self.max_len, :self.max_len] = A_con_words * sim
        H_fcon = self.gcn_dense(X_con, A_fcon_full, self.W_fcon)

        z_aff = (H_aff * asp_mask).sum(dim=1) / asp_wn
        z_con = (H_con[:, :self.max_len, :] * asp_mask).sum(dim=1) / asp_wn
        z_faff = (H_faff * asp_mask).sum(dim=1) / asp_wn
        z_fcon = (H_fcon[:, :self.max_len, :] * asp_mask).sum(dim=1) / asp_wn

        feat = torch.cat([global_bert, z_faff, z_fcon, z_aff, z_con], dim=1)
        return self.out(feat)

In [ ]:

# --------------------------
# Train / Eval
# --------------------------
def run_epoch(model, loader, optimizer=None, scheduler=None):
    train = optimizer is not None
    model.train() if train else model.eval()

    losses = []
    correct = 0
    total = 0
    all_y = []
    all_p = []

    loss_fn = nn.CrossEntropyLoss()

    for batch in loader:
        for k in batch:
            batch[k] = batch[k].to(device)

        logits = model(batch)
        y = batch["targets"]
        loss = loss_fn(logits, y)

        if train:
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            if scheduler is not None:
                scheduler.step()

        preds = torch.argmax(logits, dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)
        losses.append(loss.item())

        all_y.append(y.detach().cpu().numpy())
        all_p.append(preds.detach().cpu().numpy())

    acc = correct / max(total, 1)
    all_y = np.concatenate(all_y)
    all_p = np.concatenate(all_p)
    macro_f1 = f1_score(all_y, all_p, average="macro")
    return acc, float(np.mean(losses)), macro_f1

model = KEGCN(opt).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=opt.LEARNING_RATE)

total_steps = len(train_loader) * opt.EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=opt.NUM_WARMUP_STEPS,
    num_training_steps=total_steps
)

best_test_acc = -1.0
best_test_f1  = -1.0
best_acc_path = os.path.join(OUT_DIR, "best_by_acc.pth")
best_f1_path  = os.path.join(OUT_DIR, "best_by_f1.pth")

for epoch in range(opt.EPOCHS):
    print(f"\nEpoch {epoch+1}/{opt.EPOCHS}")
    print("-"*30)

    tr_acc, tr_loss, tr_f1 = run_epoch(model, train_loader, optimizer, scheduler)
    dv_acc, dv_loss, dv_f1 = run_epoch(model, dev_loader)
    te_acc, te_loss, te_f1 = run_epoch(model, test_loader)

    print(f"Train loss {tr_loss:.4f} acc {tr_acc:.4f} f1 {tr_f1:.4f}")
    print(f"Val   loss {dv_loss:.4f} acc {dv_acc:.4f} f1 {dv_f1:.4f}")
    print(f"TEST  acc  {te_acc:.4f} f1 {te_f1:.4f}")

    if te_acc > best_test_acc:
        best_test_acc = te_acc
        torch.save({
            "epoch": epoch + 1,
            "test_acc": te_acc,
            "test_f1": te_f1,
            "model_state_dict": model.state_dict(),
            "opt": vars(opt),
        }, best_acc_path)
        print("✅ saved best-by-acc:", best_acc_path)

    if te_f1 > best_test_f1:
        best_test_f1 = te_f1
        torch.save({
            "epoch": epoch + 1,
            "test_acc": te_acc,
            "test_f1": te_f1,
            "model_state_dict": model.state_dict(),
            "opt": vars(opt),
        }, best_f1_path)
        print("✅ saved best-by-f1:", best_f1_path)

print("\nBest TEST acc:", best_test_acc)
print("Best TEST f1 :", best_test_f1)
print("Best-by-acc checkpoint:", best_acc_path)
print("Best-by-f1  checkpoint:", best_f1_path)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]


Epoch 1/30
------------------------------
Train loss 0.7397 acc 0.6782 f1 0.5354
Val   loss 0.6988 acc 0.7086 f1 0.6668
TEST  acc  0.7146 f1 0.6780
✅ saved best-by-acc: /kaggle/working/dpfn_runs_updated/best_by_acc.pth
✅ saved best-by-f1: /kaggle/working/dpfn_runs_updated/best_by_f1.pth

Epoch 2/30
------------------------------
Train loss 0.4988 acc 0.8031 f1 0.7681
Val   loss 0.6846 acc 0.7308 f1 0.6857
TEST  acc  0.7464 f1 0.7062
✅ saved best-by-acc: /kaggle/working/dpfn_runs_updated/best_by_acc.pth
✅ saved best-by-f1: /kaggle/working/dpfn_runs_updated/best_by_f1.pth

Epoch 3/30
------------------------------
Train loss 0.2898 acc 0.8990 f1 0.8760
Val   loss 0.7351 acc 0.7415 f1 0.6710
TEST  acc  0.7570 f1 0.7053
✅ saved best-by-acc: /kaggle/working/dpfn_runs_updated/best_by_acc.pth

Epoch 4/30
------------------------------
Train loss 0.1671 acc 0.9446 f1 0.9321
Val   loss 0.9713 acc 0.7282 f1 0.6672
TEST  acc  0.7637 f1 0.7228
✅ saved best-by-acc: /kaggle/working/dpfn_runs_update

In [14]:
print(test_df["sentiment"].value_counts().sort_index())


sentiment
0    113
1    607
2    317
Name: count, dtype: int64


In [15]:
import os
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BEST_ACC = best_acc_path 
BEST_F1  = best_f1_path 
OUT_DIR  = "/kaggle/working/final_eval"
os.makedirs(OUT_DIR, exist_ok=True)

CLASS_NAMES = ["negative", "neutral", "positive"]

# ---------- helpers ----------
@torch.no_grad()
def infer_all(model, loader):
    model.eval()
    ys, ps = [], []
    for batch in loader:
        for k in batch:
            batch[k] = batch[k].to(device)
        logits = model(batch)
        preds = torch.argmax(logits, dim=1)
        ys.append(batch["targets"].cpu().numpy())
        ps.append(preds.cpu().numpy())
    return np.concatenate(ys), np.concatenate(ps)

def save_confusion(y_true, y_pred, tag):

    labels = [0, 1, 2]  # fixed order: negative, neutral, positive
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    # cm = confusion_matrix(y_true, y_pred)

    # CSV
    df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
    csv_path = os.path.join(OUT_DIR, f"confusion_{tag}.csv")
    df.to_csv(csv_path)

    # PNG
    png_path = os.path.join(OUT_DIR, f"confusion_{tag}.png")
    plt.figure(figsize=(5,5))
    plt.imshow(cm)
    plt.title(f"Confusion Matrix ({tag})")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.xticks(range(3), CLASS_NAMES, rotation=45)
    plt.yticks(range(3), CLASS_NAMES)
    plt.colorbar()

    for i in range(3):
        for j in range(3):
            plt.text(j, i, cm[i, j], ha="center", va="center")

    plt.tight_layout()
    plt.savefig(png_path, dpi=200)
    plt.close()

    return csv_path, png_path

# ---------- evaluate function ----------
def evaluate_checkpoint(ckpt_path, tag):
    print(f"\nEvaluating: {ckpt_path}")

    ckpt = torch.load(ckpt_path, map_location=device)
    model = KEGCN(opt).to(device)
    model.load_state_dict(ckpt["model_state_dict"], strict=True)

    y_true, y_pred = infer_all(model, test_loader)

    acc = accuracy_score(y_true, y_pred)
    mf1 = f1_score(y_true, y_pred, average="macro")

    print(f"[{tag}] TEST Accuracy = {acc:.4f}")
    print(f"[{tag}] TEST Macro-F1 = {mf1:.4f}")

    csv_path, png_path = save_confusion(y_true, y_pred, tag)
    print(f"Saved CM CSV: {csv_path}")
    print(f"Saved CM PNG: {png_path}")

    return acc, mf1

# ---------- RUN ----------
acc1, f11 = evaluate_checkpoint(BEST_ACC, "best_by_acc")
acc2, f12 = evaluate_checkpoint(BEST_F1,  "best_by_f1")

print("\n=== FINAL SUMMARY ===")
print(f"best_by_acc → acc={acc1:.4f}, f1={f11:.4f}")
print(f"best_by_f1  → acc={acc2:.4f}, f1={f12:.4f}")
print("Outputs saved in:", OUT_DIR)



Evaluating: /kaggle/working/dpfn_runs_updated/best_by_acc.pth
[best_by_acc] TEST Accuracy = 0.7850
[best_by_acc] TEST Macro-F1 = 0.7491
Saved CM CSV: /kaggle/working/final_eval/confusion_best_by_acc.csv
Saved CM PNG: /kaggle/working/final_eval/confusion_best_by_acc.png

Evaluating: /kaggle/working/dpfn_runs_updated/best_by_f1.pth
[best_by_f1] TEST Accuracy = 0.7850
[best_by_f1] TEST Macro-F1 = 0.7491
Saved CM CSV: /kaggle/working/final_eval/confusion_best_by_f1.csv
Saved CM PNG: /kaggle/working/final_eval/confusion_best_by_f1.png

=== FINAL SUMMARY ===
best_by_acc → acc=0.7850, f1=0.7491
best_by_f1  → acc=0.7850, f1=0.7491
Outputs saved in: /kaggle/working/final_eval
